In [2]:
import numpy as np
from scipy.special import roots_hermitenorm, eval_hermitenorm, factorial
from scipy.optimize import brentq
import pandas as pd

def gaussian_expectation(f, n_quad=200):
    """
    Compute E[f(Z)] for Z ~ N(0,1)
    """
    x, w = roots_hermitenorm(n_quad)
    return np.sum(w * f(x)) / np.sqrt(2 * np.pi)

def normalize_transform(f, n_quad=200):
    """
    Return a centered variance-1 version of f
    """

    mean = gaussian_expectation(f, n_quad)
    var = gaussian_expectation(
        lambda z: (f(z)-mean)**2,
        n_quad
    )
    sd = np.sqrt(var)

    return lambda z: (f(z)-mean)/sd

def get_hermite_coeff(f, k, n_quad=200):
    """
    Orthonormal Hermite coefficient

    c_k = E[f(Z)He_k(Z)] / sqrt(k!)
    where Z ~ N(0, 1)
    """
    x, w = roots_hermitenorm(n_quad)

    expectation = (
        np.sum(w * f(x) * eval_hermitenorm(k, x))
        / np.sqrt(2 * np.pi)
    )

    return expectation / np.sqrt(factorial(k, exact=False))

def hermite_distribution(f, max_order=20, n_quad=150):
    """
    Calculate Hermite coefficients and energies for a
    centered, variance-normalized transform.

    Returns
    dict containing:
        coefficients
        energies
        nonlinear_energy
        first_order_energy
        captured_energy
    """

    g = normalize_transform(f, n_quad)

    coeffs = np.array([
        get_hermite_coeff(g, k, n_quad)
        for k in range(max_order + 1)
    ])

    energies = coeffs**2

    return {
        "coefficients": coeffs,
        "energies": energies,
        "first_order_energy": energies[1],
        "nonlinear_energy": 1 - energies[1],
        "captured_energy": np.sum(energies)
    }

def find_strength(
    transform,
    target_nonlinear_energy,
    strength_bounds,
    max_order=20,
    n_quad=150
):
    """
    Find theta such that

        1 - rho^2 = target_nonlinear_energy

    where rho = Corr(Z, T_theta(Z)).

    result : dict
        strength
        nonlinear_energy
        first_order_energy
        coefficients
        energies
        captured_energy
    """

    def objective(strength):

        f = lambda z: transform(z, strength)

        result = hermite_distribution(
            f,
            max_order=max_order,
            n_quad=n_quad
        )

        return (
            result["nonlinear_energy"]
            - target_nonlinear_energy
        )

    strength = brentq(
        objective,
        strength_bounds[0],
        strength_bounds[1]
    )

    f = lambda z: transform(z, strength)

    result = hermite_distribution(
        f,
        max_order=max_order,
        n_quad=n_quad
    )

    result["strength"] = strength

    return result

def cubic(z, strength):
    return z + strength * z**3


def quintic(z, strength):
    return z + strength * z**5


def exponential(z, strength):
    if strength == 0:
        return z
    return np.expm1(strength * z) / strength


def quad_cubic(z, a):
    z = np.asarray(z, dtype=float)
    return z + a * z**2 + (a**2 / 3.0) * z**3


for transform in [cubic, quintic, exponential, quad_cubic]:     
    print(transform)  
    for target in [0.05, 0.1, 0.15, 0.2, 0.3]:
            
        result = find_strength(
            transform=transform,
            target_nonlinear_energy=target,
            strength_bounds=(0.0, 5),
            max_order=15
        )

        print("Required strength:", result["strength"])

def get_hermite_coeffs(transform_fn, strength, k_max=5, n_quad=200):
    x, w = roots_hermitenorm(n_quad)
    w = w / np.sqrt(2 * np.pi)

    y = transform_fn(x, strength)

    y_mean = np.sum(w * y)
    y_sd = np.sqrt(np.sum(w * (y - y_mean)**2))
    y = (y - y_mean) / y_sd

    coeffs = []

    for k in range(k_max + 1):
        c_k = np.sum(
            w * y * eval_hermitenorm(k, x)
        ) / np.sqrt(factorial(k))

        coeffs.append(c_k)

    return np.array(coeffs)

def sinh(z, a):
    if np.isclose(a, 0.0):
        return z
    return np.sinh(a * z) / a


def tanh(z, a):
    return z + a * np.tanh(z)

TRANSFORMS = {
    "cubic": cubic,
    "sinh": sinh,
    "exp": exponential,
    "tanh": tanh,
}


STRENGTHS = {
    "cubic": [0.0, 0.10, 0.25, 0.50, 1.00],
    "sinh":  [0.0, 0.35, 0.70, 1.00, 1.30],
    "exp":   [0.0, 0.20, 0.40, 0.60, 0.80],
    "tanh":  [0.0, 0.25, 0.50, 1.00, 2.00],
}

hermite_rows = []

for transform_name, strengths in STRENGTHS.items():
    for strength in strengths:

        coeffs = get_hermite_coeffs(
            TRANSFORMS[transform_name],
            strength
        )

        row = {
            "transform": transform_name,
            "strength": strength
        }

        for k, c in enumerate(coeffs):
            row[f"c{k}"] = c
            row[f"energy{k}"] = c**2

        row["higher_order_energy"] = np.sum(coeffs[2:]**2)

        row["weighted_higher_order_energy"] = np.sum(
            np.arange(2, len(coeffs)) * coeffs[2:]**2
        )

        hermite_rows.append(row)


hermite_results = pd.DataFrame(hermite_rows)

print(hermite_results)

hermite_results.to_csv(
    "../results/hermite_metrics.csv",
    index=False
)

<function cubic at 0x000001C334FCA480>
Required strength: 0.13025788811458458
Required strength: 0.22996598285207484
Required strength: 0.3532380757938138
Required strength: 0.5265986323710918
Required strength: 1.3483314773547856
<function quintic at 0x000001C334FCA520>
Required strength: 0.009807620374411513
Required strength: 0.015267537520168045
Required strength: 0.02046040925828413
Required strength: 0.02586276137841687
Required strength: 0.038479589734596774
<function exponential at 0x000001C334FC9120>
Required strength: 0.3189425386491181
Required strength: 0.45513350013398407
Required strength: 0.5627497349482242
Required strength: 0.656385717230547
Required strength: 0.8218707884650162
<function quad_cubic at 0x000001C334FCA5C0>
Required strength: 0.16592806829167261
Required strength: 0.2476383907709284
Required strength: 0.3223790465869065
Required strength: 0.3994894169143198
Required strength: 0.591223660701687
   transform  strength            c0       energy0        c1 

In [3]:
TRANSFORMS = {
    "cubic": cubic,
    "quintic": quintic,
    "exp": exponential,
    "quad": quad_cubic,
}


STRENGTHS = {
    "cubic": [0.1302579, 0.22996560, 0.3532381, 0.5265986, 1.3483315],
    "quintic":  [0.0098076, 0.0152675, 0.0204604, 0.0258628, 0.038476],
    "exp":   [0.3189425, 0.4551335, 0.5627497, 0.6563857, 0.8218707],
    "quad":  [0.1659280, 0.2476384, 0.3223790, 0.3994894, 0.5912237],
}

hermite_rows = []

for transform_name, strengths in STRENGTHS.items():
    for strength in strengths:

        coeffs = get_hermite_coeffs(
            TRANSFORMS[transform_name],
            strength
        )

        row = {
            "transform": transform_name,
            "strength": strength
        }

        for k, c in enumerate(coeffs):
            row[f"c{k}"] = c
            row[f"energy{k}"] = c**2

        row["higher_order_energy"] = np.sum(coeffs[2:]**2)

        row["weighted_higher_order_energy"] = np.sum(
            np.arange(2, len(coeffs)) * coeffs[2:]**2
        )

        hermite_rows.append(row)


hermite_results = pd.DataFrame(hermite_rows)

print(hermite_results)

hermite_results.to_csv(
    "../results/matched_hermite_metrics.csv",
    index=False
)

   transform  strength            c0       energy0        c1   energy1  \
0      cubic  0.130258  0.000000e+00  0.000000e+00  0.974679  0.950000   
1      cubic  0.229966  0.000000e+00  0.000000e+00  0.948683  0.900000   
2      cubic  0.353238  2.775558e-17  7.703720e-34  0.921954  0.850000   
3      cubic  0.526599 -2.775558e-17  7.703720e-34  0.894427  0.800000   
4      cubic  1.348332 -2.775558e-17  7.703720e-34  0.836660  0.700000   
5    quintic  0.009808  0.000000e+00  0.000000e+00  0.974680  0.950000   
6    quintic  0.015267  0.000000e+00  0.000000e+00  0.948683  0.900000   
7    quintic  0.020460  0.000000e+00  0.000000e+00  0.921954  0.850000   
8    quintic  0.025863  0.000000e+00  0.000000e+00  0.894427  0.800000   
9    quintic  0.038476 -8.326673e-17  6.933348e-33  0.836675  0.700025   
10       exp  0.318943 -5.551115e-17  3.081488e-33  0.974679  0.950000   
11       exp  0.455134 -1.110223e-16  1.232595e-32  0.948683  0.900000   
12       exp  0.562750 -8.326673e-17  

In [4]:
#Inverse Metrics
def inverse_polynomial_spectrum(
    transform_fn,
    strength,
    max_order=10,
    n_quad=200
):
    z, w = roots_hermitenorm(n_quad)
    w = w / np.sqrt(2 * np.pi)

    x = transform_fn(z, strength)

    x_mean = np.sum(w * x)
    x_sd = np.sqrt(np.sum(w * (x - x_mean)**2))
    x = (x - x_mean) / x_sd

    polynomial_basis = np.column_stack([
        x**k for k in range(max_order + 1)
    ])

    weighted_basis = np.sqrt(w)[:, None] * polynomial_basis

    Q, _ = np.linalg.qr(
        weighted_basis,
        mode="reduced"
    )

    weighted_target = np.sqrt(w) * z

    coeffs = Q.T @ weighted_target

    return coeffs

TRANSFORMS = {
    "cubic": cubic,
    "quintic": quintic,
    "exp": exponential,
    "quad": quad_cubic,
}


STRENGTHS = {
    "cubic": [0.1302579, 0.22996560, 0.3532381, 0.5265986, 1.3483315],
    "quintic":  [0.0098076, 0.0152675, 0.0204604, 0.0258628, 0.038476],
    "exp":   [0.3189425, 0.4551335, 0.5627497, 0.6563857, 0.8218707],
    "quad":  [0.1659280, 0.2476384, 0.3223790, 0.3994894, 0.5912237],
}

inverse_rows = []

for transform_name, strengths in STRENGTHS.items():
    for strength in strengths:

        coeffs = inverse_polynomial_spectrum(
            TRANSFORMS[transform_name],
            strength,
            max_order=10
        )

        row = {
            "transform": transform_name,
            "strength": strength
        }

        for k, c in enumerate(coeffs):
            row[f"inv_c{k}"] = c
            row[f"inv_energy{k}"] = c**2

        row["inv_higher_order_energy"] = (
            1 - coeffs[1]**2
        )

        row["inv_weighted_higher_order_energy"] = np.sum(
            np.arange(2, len(coeffs)) * coeffs[2:]**2
        )

        for degree in [1, 2, 3, 5, 8, 10]:
            if degree < len(coeffs):
                row[f"inv_residual_degree{degree}"] = (
                    1 - np.sum(coeffs[:degree + 1]**2)
                )

        inverse_rows.append(row)


inverse_results = pd.DataFrame(inverse_rows)

print(inverse_results)

inverse_results.to_csv(
    "../results/inverse_polynomial_metrics.csv",
    index=False
)

   transform  strength        inv_c0   inv_energy0    inv_c1  inv_energy1  \
0      cubic  0.130258  3.732269e-18  1.392983e-35  0.974679     0.950000   
1      cubic  0.229966  3.732269e-18  1.392983e-35  0.948683     0.900000   
2      cubic  0.353238  3.732269e-18  1.392983e-35  0.921954     0.850000   
3      cubic  0.526599  3.732269e-18  1.392983e-35  0.894427     0.800000   
4      cubic  1.348332  3.732269e-18  1.392983e-35  0.836660     0.700000   
5    quintic  0.009808  3.732269e-18  1.392983e-35  0.974680     0.950000   
6    quintic  0.015267  3.732269e-18  1.392983e-35  0.948683     0.900000   
7    quintic  0.020460  3.732269e-18  1.392983e-35  0.921954     0.850000   
8    quintic  0.025863  3.732269e-18  1.392983e-35  0.894427     0.800000   
9    quintic  0.038476  3.732269e-18  1.392983e-35  0.836675     0.700025   
10       exp  0.318943  3.732269e-18  1.392983e-35  0.974679     0.950000   
11       exp  0.455134  3.732269e-18  1.392983e-35  0.948683     0.900000   

In [5]:
#Yao distribution metrics
def yeojohnson(z, lam):
    z = np.asarray(z, dtype=float)
    out = np.empty_like(z)

    positive = z >= 0
    negative = ~positive

    if np.isclose(lam, 0.0):
        out[positive] = np.log1p(z[positive])
    else:
        out[positive] = (
            (z[positive] + 1.0)**lam - 1.0
        ) / lam

    if np.isclose(lam, 2.0):
        out[negative] = -np.log1p(-z[negative])
    else:
        out[negative] = -(
            (1.0 - z[negative])**(2.0 - lam) - 1.0
        ) / (2.0 - lam)

    return out

for target in [0.05, 0.1, 0.15, 0.2, 0.3]:
        result = find_strength(
            transform=yeojohnson,
            target_nonlinear_energy=target,
            strength_bounds=(1, 5),
            max_order=15
        )

        print("Required strength:", result["strength"])

Required strength: 1.5286430904875867
Required strength: 1.7738809893796312
Required strength: 1.983052891785237
Required strength: 2.1798741774597445
Required strength: 2.572385937762416


In [6]:

holdout_hermite_rows = []

holdout_strengths = [1.5286431, 1.7738810, 1.9830529, 2.1798742, 2.5723859]
for strength in holdout_strengths:
        coeffs = get_hermite_coeffs(
            yeojohnson,
            strength
        )

        row = {
            "transform": "yeo",
            "strength": strength
        }

        for k, c in enumerate(coeffs):
            row[f"c{k}"] = c
            row[f"energy{k}"] = c**2

        row["higher_order_energy"] = np.sum(coeffs[2:]**2)

        row["weighted_higher_order_energy"] = np.sum(
            np.arange(2, len(coeffs)) * coeffs[2:]**2
        )

        holdout_hermite_rows.append(row)


holdout_hermite_results = pd.DataFrame(holdout_hermite_rows)
holdout_inverse_rows = []
for strength in holdout_strengths:

        coeffs = inverse_polynomial_spectrum(
            yeojohnson,
            strength,
            max_order=10
        )

        row = {
            "transform": "yeo",
            "strength": strength
        }

        for k, c in enumerate(coeffs):
            row[f"inv_c{k}"] = c
            row[f"inv_energy{k}"] = c**2

        row["inv_higher_order_energy"] = (
            1 - coeffs[1]**2
        )

        row["inv_weighted_higher_order_energy"] = np.sum(
            np.arange(2, len(coeffs)) * coeffs[2:]**2
        )

        for degree in [1, 2, 3, 5, 8, 10]:
            if degree < len(coeffs):
                row[f"inv_residual_degree{degree}"] = (
                    1 - np.sum(coeffs[:degree + 1]**2)
                )

        holdout_inverse_rows.append(row)


holdout_inverse_results = pd.DataFrame(holdout_inverse_rows)

holdout_metrics = holdout_hermite_results.merge(
    holdout_inverse_results,
    on=["transform", "strength"],
    how="inner",
    validate="one_to_one"
)
holdout_metrics.to_csv(
    "../results/holdout_yeo_metrics.csv",
    index=False
)

print(holdout_metrics)

  transform  strength            c0       energy0        c1   energy1  \
0       yeo  1.528643 -8.326673e-17  6.933348e-33  0.974679  0.950000   
1       yeo  1.773881 -1.110223e-16  1.232595e-32  0.948683  0.899999   
2       yeo  1.983053 -1.665335e-16  2.773339e-32  0.921954  0.849999   
3       yeo  2.179874 -8.326673e-17  6.933348e-33  0.894427  0.799999   
4       yeo  2.572386 -1.387779e-16  1.925930e-32  0.836660  0.699999   

         c2   energy2        c3   energy3  ...   inv_c10  inv_energy10  \
0  0.220821  0.048762  0.028810  0.000830  ...  0.001731      0.000003   
1  0.309727  0.095931  0.058795  0.003457  ... -0.004257      0.000018   
2  0.375827  0.141246  0.090009  0.008102  ...  0.005231      0.000027   
3  0.429420  0.184402  0.122492  0.015004  ...  0.027435      0.000753   
4  0.512567  0.262725  0.191306  0.036598  ...  0.070386      0.004954   

   inv_higher_order_energy  inv_weighted_higher_order_energy  \
0                 0.050000                          